# Checkout the Canteen - Workflow Tất Cả Trong Một

Notebook này là entrypoint duy nhất cho project: local, Google Colab, Kaggle và Google Drive Desktop.

Dùng notebook này để:

- cài môi trường và kiểm tra GPU;
- đọc/audit data;
- package và sync artifact lên Google Drive Desktop;
- train classifier 11 món;
- train YOLO food-region detector;
- train YOLO egg/fish detector;
- đọc report train;
- chạy Grad-CAM;
- chạy Demo App và Data IDE.

Quy ước data sạch:

```text
data/reviewed/                  nguồn tin cậy đã review thủ công
data/classification/            train/val/test được sinh ra cho classifier
data/detection/egg_fish_shared/ YOLO detector dataset đã thêm hard-negative an toàn
outputs/cloud/*.zip             file đưa lên Drive/Kaggle/Colab
models/*.pt                     model đã train
```

Archived Roboflow tray datasets are CC BY 4.0 sources. The archive stays immutable, and this notebook trains only from generated clean data. The target deployment is the University of Economics Ho Chi Minh City (UEH) canteen.


## 0. Chọn môi trường và cấu hình chung

Sửa các biến trong cell dưới nếu cần. Thường chỉ cần đổi `ENV`, `CLASSIFIER_ARCH`, `YOLO_MODEL`, `EPOCHS`.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, textwrap, zipfile
from datetime import datetime

ENV = "auto"  # auto | local | colab | kaggle
REPO_URL = "https://github.com/Vo-Minh-Tri1412/cnn-food-recognition.git"
BRANCH = "codex/colab-kaggle-workflow"

# Classifier choices: mobilenet_v3_small, mobilenet_v3_large, efficientnet_b0, efficientnet_b2, resnet18, resnet50
CLASSIFIER_ARCH = "efficientnet_b2"
CLASSIFIER_EPOCHS = 8
CLASSIFIER_BATCH = 16
CLASSIFIER_IMAGE_SIZE = 260
CLASSIFIER_AUGMENTATION = "light"  # none | light | medium | strong
CLASSIFIER_LABEL_SMOOTHING = 0.05
CLASSIFIER_LOSS = "cross_entropy"  # cross_entropy | focal
CLASSIFIER_FOCAL_GAMMA = 2.0
CLASSIFIER_SAMPLER = "shuffle"  # shuffle | balanced
CLASSIFIER_OVERSAMPLE = {"canh_chua_khong_ca": 2.5}
CLASSIFIER_CLASS_AUGMENTATIONS = {"canh_chua_khong_ca": "strong"}

# Food-region YOLO: yolo11s is the recommended Colab baseline; use yolo11n for smoke/low-latency deployment
REGION_YOLO_MODEL = "yolo11s.pt"
REGION_YOLO_EPOCHS = 100
REGION_YOLO_IMGSZ = 640
REGION_YOLO_BATCH = 16
REGION_YOLO_PATIENCE = 20

# Egg/fish YOLO choices: yolo11n.pt, yolo11s.pt, yolo11m.pt
YOLO_MODEL = "yolo11s.pt"
YOLO_EPOCHS = 100
YOLO_IMGSZ = 640
YOLO_BATCH = 16
YOLO_PATIENCE = 20

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

def detect_env():
    if ENV != "auto":
        return ENV
    try:
        import google.colab  # type: ignore
        return "colab"
    except Exception:
        pass
    if Path("/kaggle/working").exists():
        return "kaggle"
    return "local"

ENV_NAME = detect_env()
print("ENV_NAME =", ENV_NAME)
print("RUN_STAMP =", RUN_STAMP)

## 1. Clone/pull repo và cài dependency

- Local: dùng repo đang mở.
- Colab/Kaggle: clone branch từ GitHub về runtime.

In [ ]:
def run(cmd, cwd=None, check=True):
    cmd = [str(x) for x in cmd]
    print("\n$", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with code {proc.returncode}: {' '.join(cmd)}")
    return proc

if ENV_NAME == "colab":
    WORK_ROOT = Path("/content")
    PROJECT_ROOT = WORK_ROOT / "cnn-food-recognition"
elif ENV_NAME == "kaggle":
    WORK_ROOT = Path("/kaggle/working")
    PROJECT_ROOT = WORK_ROOT / "cnn-food-recognition"
else:
    PROJECT_ROOT = Path.cwd()
    WORK_ROOT = PROJECT_ROOT

if ENV_NAME in {"colab", "kaggle"}:
    if not PROJECT_ROOT.exists():
        run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_ROOT)])
    else:
        run(["git", "fetch", "origin"], cwd=PROJECT_ROOT, check=False)
        run(["git", "checkout", BRANCH], cwd=PROJECT_ROOT, check=False)
        run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_ROOT, check=False)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("WORK_ROOT =", WORK_ROOT)

In [ ]:
# Chạy cell này sau khi clone/pull. Trên local nếu .venv đã có sẵn thì có thể bỏ qua.
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=PROJECT_ROOT)

## 2. Kiểm tra GPU và import thư viện

Nếu Colab hiện CPU/no cuda thì vào `Runtime -> Change runtime type -> GPU`. TPU không phải CUDA nên PyTorch/Ultralytics sẽ hiện no cuda.

In [ ]:
import importlib.util
import torch
print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
if importlib.util.find_spec("ultralytics") is None:
    print("Ultralytics is missing; installing it now...")
    run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3,<9"], cwd=PROJECT_ROOT)
import ultralytics
print("Ultralytics:", ultralytics.__version__)

## 3. Google Drive / Kaggle input

Colab: mount Drive và tìm zip trong `MyDrive/canteen_checkout/datasets/`.

Kaggle: add Dataset vào notebook, zip nằm trong `/kaggle/input/...`.

Local: dùng `outputs/cloud/` hoặc `data/` trong repo.
Required dataset artifacts:

```text
classification.zip
food_regions_yolo.zip
egg_fish_shared_yolo.zip  # preferred; egg_fish_yolo.zip is the fallback
```

For Colab, place them under `MyDrive/canteen_checkout/datasets/`. For Kaggle, attach a Dataset containing the same files. Section 5 creates and publishes these artifacts from the local immutable data sources.


In [ ]:
DRIVE_ROOT = None
if ENV_NAME == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/canteen_checkout")
elif ENV_NAME == "local":
    candidates = [Path("G:/My Drive/canteen_checkout"), Path("G:/MyDrive/canteen_checkout"), Path.home() / "Google Drive" / "My Drive" / "canteen_checkout"]
    DRIVE_ROOT = next((p for p in candidates if p.exists()), None)
else:
    DRIVE_ROOT = None
print("DRIVE_ROOT =", DRIVE_ROOT)

RUNTIME_DATA = WORK_ROOT / "canteen_checkout_data"

def count_images(root):
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    root = Path(root)
    return sum(1 for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts) if root.exists() else 0

def find_file(filename):
    candidates = []
    if DRIVE_ROOT:
        candidates.append(DRIVE_ROOT / "datasets" / filename)
    candidates.extend([WORK_ROOT / filename, PROJECT_ROOT / "outputs" / "cloud" / filename])
    if ENV_NAME == "kaggle":
        candidates.extend(Path("/kaggle/input").rglob(filename))
    return next((p for p in candidates if p.exists()), None)

def require_input_file(filename):
    found = find_file(filename)
    if found is not None:
        return found
    if ENV_NAME == "colab":
        from google.colab import files
        print(f"{filename} was not found in Drive. Select it in the upload dialog.")
        uploaded = files.upload()
        if filename in uploaded:
            target = WORK_ROOT / filename
            if not target.exists():
                target.write_bytes(uploaded[filename])
            return target
    raise FileNotFoundError(
        f"{filename} was not found. Put it in MyDrive/canteen_checkout/datasets/ "
        "or attach/upload the file to this runtime."
    )

def unzip_to_runtime(zip_path, runtime_root, expected_folder=None, marker=None):
    runtime_root = Path(runtime_root)
    runtime_root.mkdir(parents=True, exist_ok=True)
    if expected_folder and (runtime_root / expected_folder).exists():
        shutil.rmtree(runtime_root / expected_folder)
    with zipfile.ZipFile(zip_path) as zf:
        top_folders = sorted({Path(name).parts[0] for name in zf.namelist() if Path(name).parts})
        zf.extractall(runtime_root)
    if expected_folder and (runtime_root / expected_folder).exists():
        return runtime_root / expected_folder
    if marker:
        for folder_name in top_folders:
            candidate = runtime_root / folder_name
            if candidate.is_dir() and (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(f"No extracted dataset folder found for {zip_path}")

## 4. Data status and clean contract

This cell is read-only. Training and demo workflows use `reviewed`, `classification`, `food_regions`, `egg_fish_shared`, and `outputs/cloud`; the Roboflow archive is only an immutable build source.


In [ ]:
def count_images(root):
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    root = Path(root)
    return sum(1 for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts) if root.exists() else 0

for rel in ["data/reviewed", "data/classification", "data/detection/food_regions", "data/detection/egg_fish", "data/detection/egg_fish_shared", "data/inbox/review", "data/quarantine"]:
    p = PROJECT_ROOT / rel
    print(f"{rel:34s}", count_images(p), "images", "OK" if p.exists() else "missing")

reviewed = PROJECT_ROOT / "data" / "reviewed"
if reviewed.exists():
    print("\nReviewed by class:")
    for d in sorted([x for x in reviewed.iterdir() if x.is_dir()]):
        print(f"  {d.name:24s} {count_images(d)}")

## 5. Local: build/package/sync data lên Drive một phát

Chạy section này trên máy local khi muốn cập nhật Google Drive Desktop. Trên Colab/Kaggle thì bỏ qua.

In [ ]:
if ENV_NAME == "local":
    # Package classifier dataset
    run([sys.executable, "scripts/data/03_package_classification_dataset.py"], cwd=PROJECT_ROOT)

    # Build safe YOLO shared negatives, then package
    base_yolo = PROJECT_ROOT / "data" / "detection" / "egg_fish"
    if base_yolo.exists():
        run([sys.executable, "scripts/data/05_build_yolo_shared_negatives.py", "--clear", "--max-per-class", "80"], cwd=PROJECT_ROOT)
        run([
            sys.executable, "scripts/data/06_package_yolo_dataset.py",
            "--source", "data/detection/egg_fish_shared",
            "--output", "outputs/cloud/egg_fish_shared_yolo.zip",
            "--manifest", "outputs/cloud/egg_fish_shared_yolo.manifest.json",
        ], cwd=PROJECT_ROOT)
    else:
        print("Bỏ qua YOLO package: data/detection/egg_fish không tồn tại")

    # Build a clean one-class food-region dataset without changing data/archive, then package it.
    tray_archives = sorted((PROJECT_ROOT / "data" / "archive").glob("raw_tray_datasets_*"))
    if tray_archives:
        run([sys.executable, "scripts/data/08_build_food_region_dataset.py", "--clear"], cwd=PROJECT_ROOT)
        run([
            sys.executable, "scripts/data/06_package_yolo_dataset.py",
            "--source", "data/detection/food_regions",
            "--output", "outputs/cloud/food_regions_yolo.zip",
            "--manifest", "outputs/cloud/food_regions_yolo.manifest.json",
        ], cwd=PROJECT_ROOT)
    else:
        print("Skip food-region package: raw_tray_datasets archive was not found")

    # One-shot publish: datasets + models + notebook/docs to Google Drive Desktop
    run([sys.executable, "scripts/cloud/01_sync_drive_artifacts.py", "--publish", "--apply"], cwd=PROJECT_ROOT)
else:
    print("Bỏ qua: section này dành cho local + Google Drive Desktop.")

## 6. Lấy classification.zip vào runtime

Cell này copy zip về runtime rồi unzip local. Không train trực tiếp trên Drive vì nhiều file nhỏ sẽ chậm.

In [ ]:
classification_zip = require_input_file("classification.zip")
CLASSIFICATION_ROOT = unzip_to_runtime(classification_zip, RUNTIME_DATA, expected_folder="classification")
print("classification_zip =", classification_zip)
print("CLASSIFICATION_ROOT =", CLASSIFICATION_ROOT)
for split in ["train", "val", "test"]:
    print(split, count_images(CLASSIFICATION_ROOT / split))

## 7. Audit classification dataset

Audit = kiểm kê/kiểm tra dataset trước khi train: có đủ split/class không, có conflict duplicate nghiêm trọng không.

In [ ]:
# Nếu script audit tồn tại thì chạy, nếu không thì in count cơ bản.
audit_script = PROJECT_ROOT / "scripts" / "data" / "02_audit_dataset_conflicts.py"
if audit_script.exists():
    run([sys.executable, str(audit_script), "--root", str(CLASSIFICATION_ROOT), "--phash-threshold", "4"], cwd=PROJECT_ROOT, check=False)
else:
    for split in ["train", "val", "test"]:
        print(split)
        for d in sorted((CLASSIFICATION_ROOT / split).iterdir()):
            if d.is_dir():
                print(" ", d.name, count_images(d))

## 8. Train classifier 11 món

Đổi model ở cell 0 bằng `CLASSIFIER_ARCH`. Kết quả nằm trong `RUN_ROOT/models` và `RUN_ROOT/outputs/reports`.

In [ ]:
RUN_ROOT = WORK_ROOT / "canteen_checkout_runs" / f"classifier_{RUN_STAMP}"
MODEL_OUT = RUN_ROOT / "models" / "dish_classifier.pt"
REPORT_OUT = RUN_ROOT / "outputs" / "reports"
os.environ["CANTEEN_MODEL_DIR"] = str(RUN_ROOT / "models")
os.environ["CANTEEN_OUTPUTS_DIR"] = str(RUN_ROOT / "outputs")

cmd = [
    sys.executable, "scripts/train/01_train_classifier.py",
    "--data", str(CLASSIFICATION_ROOT),
    "--model-out", str(MODEL_OUT),
    "--epochs", str(CLASSIFIER_EPOCHS),
    "--batch-size", str(CLASSIFIER_BATCH),
    "--image-size", str(CLASSIFIER_IMAGE_SIZE),
    "--arch", CLASSIFIER_ARCH,
    "--augmentation", CLASSIFIER_AUGMENTATION,
    "--label-smoothing", str(CLASSIFIER_LABEL_SMOOTHING),
    "--loss", CLASSIFIER_LOSS,
    "--focal-gamma", str(CLASSIFIER_FOCAL_GAMMA),
    "--sampler", CLASSIFIER_SAMPLER,
]
for class_name, factor in CLASSIFIER_OVERSAMPLE.items():
    cmd += ["--oversample-class", f"{class_name}:{factor}"]
for class_name, augmentation in CLASSIFIER_CLASS_AUGMENTATIONS.items():
    cmd += ["--class-augmentation", f"{class_name}={augmentation}"]
run(cmd, cwd=PROJECT_ROOT)
print("MODEL_OUT =", MODEL_OUT)
print("REPORT_OUT =", REPORT_OUT)

## 9. Đọc report classifier

Cell này in classification report và hiển thị chart/confusion matrix nếu có.

In [ ]:
from IPython.display import Image, display
report_file = REPORT_OUT / "classification_report.txt"
if report_file.exists():
    print(report_file.read_text(encoding="utf-8"))
else:
    print("Missing", report_file)

for image_name in ["training_history.png", "confusion_matrix.png"]:
    p = REPORT_OUT / image_name
    if p.exists():
        print("\n", p)
        display(Image(filename=str(p)))

## 10. Grad-CAM classifier

Dùng để xem model đang nhìn vào đâu. Mặc định lấy các mẫu sai/yếu trong test set.

In [ ]:
from IPython.display import Image, display
gradcam_script = PROJECT_ROOT / "scripts" / "debug" / "01_gradcam_debug.py"
if gradcam_script.exists() and MODEL_OUT.exists():
    GRADCAM_OUT = RUN_ROOT / "outputs" / "gradcam"
    run([
        sys.executable, str(gradcam_script),
        "--data", str(CLASSIFICATION_ROOT),
        "--model", str(MODEL_OUT),
        "--out", str(GRADCAM_OUT),
        "--split", "test",
        "--max-samples", "32",
    ], cwd=PROJECT_ROOT, check=False)
    print("GRADCAM_OUT =", GRADCAM_OUT)
    for p in sorted(GRADCAM_OUT.rglob("*.jpg"))[:8]:
        display(Image(filename=str(p)))
else:
    print("Skip Grad-CAM: missing script or model.")

## 11. Load the food-region YOLO dataset

The one-class `food_region` dataset is generated from representative Roboflow variants with normalized annotations and cross-export deduplication. Roboflow Stretch cannot be reversed, so final evaluation must use an independent UEH tray holdout.

**Region-detector-only path on Colab:** run sections 0-3, then sections 11-13. Sections 6-10 are only needed for the dish classifier. If the zip is not in `MyDrive/canteen_checkout/datasets/`, section 11 opens an upload dialog for `food_regions_yolo.zip`.


In [ ]:
if "require_input_file" not in globals():
    raise RuntimeError("Run sections 0-3 first so Colab installs dependencies, mounts Drive, and defines shared helpers.")
region_zip = require_input_file("food_regions_yolo.zip")
region_manifest = find_file("food_regions_yolo.manifest.json")
if region_manifest:
    print(region_manifest.read_text(encoding="utf-8"))

REGION_DATA_ROOT = unzip_to_runtime(region_zip, RUNTIME_DATA, expected_folder="food_regions", marker="data.yaml")
REGION_YAML = REGION_DATA_ROOT / "data.yaml"
print("region_zip =", region_zip)
print("REGION_DATA_ROOT =", REGION_DATA_ROOT)
print(REGION_YAML.read_text(encoding="utf-8"))
for split in ["train", "valid", "test"]:
    print(split, count_images(REGION_DATA_ROOT / split / "images"), "images")


## 12. Train the food-region YOLO detector

This model only finds food regions. `yolo11s.pt` is the recommended Colab baseline for the current 365-image clean dataset. Use `yolo11n.pt` for a quick smoke test or lower latency. Do not move to `yolo11m.pt` until the UEH holdout shows that model capacity, rather than domain/data quality, is the limiting factor.


In [ ]:
REGION_RUN_ROOT = WORK_ROOT / "canteen_checkout_runs" / f"food_regions_{RUN_STAMP}"
REGION_MODEL_OUT = REGION_RUN_ROOT / "models" / "food_region_detector.pt"
os.environ["CANTEEN_MODEL_DIR"] = str(REGION_RUN_ROOT / "models")
os.environ["CANTEEN_OUTPUTS_DIR"] = str(REGION_RUN_ROOT / "outputs")
cmd = [
    sys.executable, "scripts/train/03_train_food_region_detector.py",
    "--data", str(REGION_YAML),
    "--model", REGION_YOLO_MODEL,
    "--epochs", str(REGION_YOLO_EPOCHS),
    "--imgsz", str(REGION_YOLO_IMGSZ),
    "--batch", str(REGION_YOLO_BATCH),
    "--patience", str(REGION_YOLO_PATIENCE),
    "--project", str(REGION_RUN_ROOT / "detector"),
    "--name", "train",
    "--model-out", str(REGION_MODEL_OUT),
]
run(cmd, cwd=PROJECT_ROOT)
print("REGION_MODEL_OUT =", REGION_MODEL_OUT)


## 13. Read the food-region YOLO report

Besides mAP, validate region recall, duplicate crops, and exact item count on the independent UEH tray holdout.


In [ ]:
region_summary = REGION_RUN_ROOT / "outputs" / "reports" / "food_region_detector_training_summary.json"
if region_summary.exists():
    print(region_summary.read_text(encoding="utf-8"))
else:
    print("food-region training summary was not found:", region_summary)

for p in sorted((REGION_RUN_ROOT / "detector" / "train").glob("*.png"))[:8]:
    print(p)
    display(Image(filename=str(p)))

from ultralytics import YOLO
region_test_metrics = YOLO(str(REGION_MODEL_OUT)).val(
    data=str(REGION_YAML),
    split="test",
    imgsz=REGION_YOLO_IMGSZ,
    batch=REGION_YOLO_BATCH,
    project=str(REGION_RUN_ROOT / "detector"),
    name="test_eval",
)
print(region_test_metrics.results_dict)


## 14. Lấy YOLO egg/fish dataset vào runtime

Notebook ưu tiên `egg_fish_shared_yolo.zip` vì có hard-negative từ classification reviewed.

In [ ]:
yolo_zip = find_file("egg_fish_shared_yolo.zip") or find_file("egg_fish_yolo.zip")
if yolo_zip is None:
    raise FileNotFoundError("Không tìm thấy egg_fish_shared_yolo.zip hoặc egg_fish_yolo.zip")

# Xóa folder cũ nếu có
for folder in [RUNTIME_DATA / "egg_fish_shared", RUNTIME_DATA / "egg_fish"]:
    if folder.exists():
        shutil.rmtree(folder)
YOLO_DATA_ROOT = unzip_to_runtime(yolo_zip, RUNTIME_DATA, marker="data.yaml")
YOLO_YAML = YOLO_DATA_ROOT / "data.yaml"
print("yolo_zip =", yolo_zip)
print("YOLO_DATA_ROOT =", YOLO_DATA_ROOT)
print(YOLO_YAML.read_text(encoding="utf-8"))
for split in ["train", "valid", "test"]:
    print(split, count_images(YOLO_DATA_ROOT / split / "images"), "images")

## 15. Train YOLO egg/fish detector

Detector chỉ phụ trợ fusion: `egg` cho thịt kho trứng, `fish` cho canh chua có cá. Không thay classifier.

In [ ]:
YOLO_RUN_ROOT = WORK_ROOT / "canteen_checkout_runs" / f"yolo_{RUN_STAMP}"
YOLO_MODEL_OUT = YOLO_RUN_ROOT / "models" / "egg_fish_detector.pt"
os.environ["CANTEEN_MODEL_DIR"] = str(YOLO_RUN_ROOT / "models")
os.environ["CANTEEN_OUTPUTS_DIR"] = str(YOLO_RUN_ROOT / "outputs")
cmd = [
    sys.executable, "scripts/train/02_train_yolo_detector.py",
    "--data", str(YOLO_YAML),
    "--model", YOLO_MODEL,
    "--epochs", str(YOLO_EPOCHS),
    "--imgsz", str(YOLO_IMGSZ),
    "--batch", str(YOLO_BATCH),
    "--patience", str(YOLO_PATIENCE),
    "--project", str(YOLO_RUN_ROOT / "detector"),
    "--name", "train",
    "--model-out", str(YOLO_MODEL_OUT),
]
run(cmd, cwd=PROJECT_ROOT)
print("YOLO_MODEL_OUT =", YOLO_MODEL_OUT)

## 16. Đọc report YOLO egg/fish

Xem metric summary và plot do Ultralytics sinh ra.

In [ ]:
yolo_summary = YOLO_RUN_ROOT / "outputs" / "reports" / "egg_fish_detector_training_summary.json"
# Script train có thể ghi report vào env OUTPUTS_DIR nếu được set từ classifier; nên tìm rộng hơn.
candidates = list(YOLO_RUN_ROOT.rglob("egg_fish_detector_training_summary.json")) + list((WORK_ROOT / "canteen_checkout_runs").rglob("egg_fish_detector_training_summary.json"))
if candidates:
    latest = max(candidates, key=lambda p: p.stat().st_mtime)
    print(latest.read_text(encoding="utf-8"))
else:
    print("Không tìm thấy YOLO summary json")

for p in sorted((YOLO_RUN_ROOT / "detector" / "train").glob("*.png"))[:8]:
    print(p)
    display(Image(filename=str(p)))

## 17. Lưu model/report về Google Drive

Colab: copy run và canonical model về Drive.

Local: có thể dùng section 5 hoặc script `scripts/cloud/01_sync_drive_artifacts.py --publish --apply`.

In [ ]:
if DRIVE_ROOT:
    drive_runs = DRIVE_ROOT / "runs" / RUN_STAMP
    drive_models = DRIVE_ROOT / "models"
    drive_runs.mkdir(parents=True, exist_ok=True)
    drive_models.mkdir(parents=True, exist_ok=True)
    for root in [RUN_ROOT, REGION_RUN_ROOT if 'REGION_RUN_ROOT' in globals() else None, YOLO_RUN_ROOT if 'YOLO_RUN_ROOT' in globals() else None]:
        if root and root.exists():
            shutil.copytree(root, drive_runs / root.name, dirs_exist_ok=True)
    if 'MODEL_OUT' in globals() and MODEL_OUT.exists():
        shutil.copy2(MODEL_OUT, drive_models / "dish_classifier.pt")
    class_names = (RUN_ROOT / "models" / "class_names.json") if 'RUN_ROOT' in globals() else None
    if class_names and class_names.exists():
        shutil.copy2(class_names, drive_models / "class_names.json")
    if 'REGION_MODEL_OUT' in globals() and REGION_MODEL_OUT.exists():
        shutil.copy2(REGION_MODEL_OUT, drive_models / "food_region_detector.pt")
    if 'YOLO_MODEL_OUT' in globals() and YOLO_MODEL_OUT.exists():
        shutil.copy2(YOLO_MODEL_OUT, drive_models / "egg_fish_detector.pt")
    print("Saved to", drive_runs)
    print("Canonical models in", drive_models)
else:
    print("DRIVE_ROOT not available; skip Drive save.")

## 18. Demo App

Local: mở `http://127.0.0.1:7863` hoặc port bạn chọn.

Colab: cell này cố gắng expose port bằng Colab output helper.

In [ ]:
DEMO_PORT = 7863
# Nếu muốn dùng model vừa train trong runtime, set env model dir trước khi start app.
if 'MODEL_OUT' in globals() and MODEL_OUT.exists():
    os.environ["CANTEEN_MODEL_DIR"] = str(MODEL_OUT.parent)
if 'RUN_ROOT' in globals():
    os.environ["CANTEEN_OUTPUTS_DIR"] = str(RUN_ROOT / "outputs")

print("Starting demo app on port", DEMO_PORT)
proc = subprocess.Popen([sys.executable, "scripts/apps/01_demo_checkout_app.py", "--host", "0.0.0.0", "--port", str(DEMO_PORT)], cwd=str(PROJECT_ROOT))
print("PID =", proc.pid)
if ENV_NAME == "colab":
    from google.colab import output
    output.serve_kernel_port_as_window(DEMO_PORT)
else:
    print(f"Open http://127.0.0.1:{DEMO_PORT}/")

## 19. Data IDE

Dùng trên local là tốt nhất. Colab vẫn có thể expose port, nhưng thao tác file local/Drive sẽ khó quản lý hơn.

In [ ]:
IDE_PORT = 7864
print("Starting Data IDE on port", IDE_PORT)
proc_ide = subprocess.Popen([sys.executable, "scripts/apps/02_data_ide.py", "--host", "0.0.0.0", "--port", str(IDE_PORT)], cwd=str(PROJECT_ROOT))
print("PID =", proc_ide.pid)
if ENV_NAME == "colab":
    from google.colab import output
    output.serve_kernel_port_as_window(IDE_PORT)
else:
    print(f"Open http://127.0.0.1:{IDE_PORT}/")

## 20. Lệnh nhanh local

The commands below reproduce the local clean-build, package, train, sync, and app workflow.

```powershell
# Build the clean food-region dataset from the immutable Roboflow archive
.\.venv\Scripts\python.exe scripts\data\08_build_food_region_dataset.py --dry-run
.\.venv\Scripts\python.exe scripts\data\08_build_food_region_dataset.py --clear

# Package it for Google Colab / Kaggle
.\.venv\Scripts\python.exe scripts\data\06_package_yolo_dataset.py `
  --source data\detection\food_regions `
  --output outputs\cloud\food_regions_yolo.zip `
  --manifest outputs\cloud\food_regions_yolo.manifest.json

# Optional local training; Colab should normally use yolo11s.pt
.\.venv\Scripts\python.exe scripts\train\03_train_food_region_detector.py `
  --data data\detection\food_regions\data.yaml `
  --model yolo11s.pt `
  --epochs 100 `
  --batch 16

# Publish packages, models, notebook, and docs to Google Drive Desktop
.\.venv\Scripts\python.exe scripts\cloud\01_sync_drive_artifacts.py --publish --apply

# Pull all three trained models and the latest report run from Drive
.\.venv\Scripts\python.exe scripts\cloud\01_sync_drive_artifacts.py --pull --apply

# Run local apps
.\.venv\Scripts\python.exe scripts\apps\01_demo_checkout_app.py --port 7863
.\.venv\Scripts\python.exe scripts\apps\02_data_ide.py --port 7864
```
